## 0. 환경 구성

### 1) 라이브러리 설치

In [ ]:
# uv add langchain_community faiss-cpu langchain-ollama

### 2) OpenAI 인증키 설정
https://openai.com/

In [ ]:
# ollama run qwen2.5:1.5b

# ollama pull bge-m3

In [ ]:
EMBEDDING_MODEL_NAME = "bge-m3:latest"
print(EMBEDDING_MODEL_NAME)

CHAT_MODEL_NAME = "qwen2.5:1.5b"
print(CHAT_MODEL_NAME)

bge-m3:latest
qwen2.5:0.5b


#### RAG 파이프 라인
* Load Data - Text Split - Indexing - Retrieval - Generation
* OllamaEmbeddings 사용

In [7]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings

from pprint import pprint

# 1. Load Data
loader = TextLoader("../data/taxinfo.txt", encoding="utf-8")
documents = loader.load()

print(type(documents), len(documents)) #[Document, Document]
print(type(documents[0]))
print(documents[0])

<class 'list'> 1
<class 'langchain_core.documents.base.Document'>
page_content='제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31.>
1. 「공익신탁법」에 따른 공익신탁의 이익
2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득
    가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득
    나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다). 이 경우 주택 수의 계산 및 주택임대소득의 산정 등 필요한 사항은 대통령령으로 정한다.
    다. 대통령령으로 정하는 농어가부업소득
    라. 대통령령으로 정하는 전통주의 제조에서 발생하는 소득
    마. 조림기간 5년 이상인 임지(林地)의 임목(林木)의 벌채 또는 양도로 발생하는 소득으로서 연 600만원 이하의 금액. 이 경우 조림기간 및 세액의 계산 등 필요한 사항은 대통령령으로 정한다.
    바. 대통령령으로 정하는 작물재배업에서 발생하는 소득
    사. 대통령령으로 정하는 어로어업 또는 양식어업에서 발

In [8]:

# 2️. Text Split
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = splitter.split_documents(documents)

print(len(split_docs), type(split_docs), type(split_docs[0]))
print(split_docs[0])
print('두번째 Document ====================')
print(split_docs[1])


12 <class 'list'> <class 'langchain_core.documents.base.Document'>
page_content='제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31.>
1. 「공익신탁법」에 따른 공익신탁의 이익
2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득
    가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득' metadata={'source': '../data/taxinfo.txt'}
두번째 Document ====================
page_content='가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득
    나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다). 이 경우 주택 수의 계산 및 주택임대소득의 산정 등 필요한 사항은 대통령령으로 정한다.
    다. 대통령령으로 정하는 농어가부업소득
    라. 대통령령으로 정하는 전통주의 제조에서 발생하는 소득
    마. 조림기간 5년 이상인 임지(林地)의 임목(林木)의 벌채 또는 양도로 발생하는 소득으로서

In [9]:

# 3️. Indexing (벡터 저장)
embeddings_model = OllamaEmbeddings(model=EMBEDDING_MODEL_NAME)
# vector DB에 저장
vectorstore = FAISS.from_documents(
    documents=split_docs, 
    embedding=embeddings_model
)

# 로컬 파일로 저장
vectorstore.save_local("faiss_index_ollama")

In [10]:

# 4️. Retrieval (유사 문서 검색) k: 질의와 가장 유사한 문서(청크) 6개를 찾아 반환하기
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})
# retriever변수는 VectoreStoreRetriever 객체이다.
print(type(retriever))
# **질문(쿼리)**에 대해 유사한 문서를 검색하는 역할
retrieved_docs = retriever.invoke("소득세법에서 비과세소득에 해당하는 소득은 무엇인가요?")
print(type(retrieved_docs), len(retrieved_docs))
print(type(retrieved_docs[0]))
print(retrieved_docs[0])

<class 'langchain_core.vectorstores.base.VectorStoreRetriever'>
<class 'list'> 6
<class 'langchain_core.documents.base.Document'>
page_content='제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31.>
1. 「공익신탁법」에 따른 공익신탁의 이익
2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득
    가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득' metadata={'source': '../data/taxinfo.txt'}


In [11]:

# 5️. Generation (LLM 응답 생성)
try:
    llm = ChatOllama(
        model=CHAT_MODEL_NAME,
        temperature=0.1,
        num_predict=800
    )
    print("LLM 설정 완료")
except Exception as e:
    print(f"LLM 설정 실패: {e}")
    exit(1)
    
# 유사도 검색의 결과를 str 문자열로 만들기
context = "\n\n".join([doc.page_content for doc in retrieved_docs])
print(context)

LLM 설정 완료
제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31.>
1. 「공익신탁법」에 따른 공익신탁의 이익
2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득
    가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득

가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득
    나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다). 이 경우 주택 수의 계산 및 주택임대소득의 산정 등 필요한 사항은 대통령령으로 정한다.
    다. 대통령령으로 정하는 농어가부업소득
    라. 대통령령으로 정하는 전통주의 제조에서 발생하는 소득
    마. 조림기간 5년 이상인 임지(林地)의 임목(林木)의 벌채 또는 양도로 발생하는 소득으로서 연 600만원 이하의 금액. 이 경우 조림기간 및 세액의 계산 등 필요한 사항은 대통령령으로 정한다.
    바. 대통령령으로 정하는 작물재배업에서 발생하는 소득

바. 대통령령으로 정하는 작물재배업에서 발생하는 소득
    사. 대통령령으로 정하는 어로어업 또는 양식어업에서 발생하는 소

In [12]:

response_context = llm.invoke(f"소득세법에서 비과세소득에 해당하는 소득은 무엇인가요? \
                              관련 정보: {context}")
print('context 적용한 결과')
pprint(response_context.content)

context 적용한 결과
('소득세법에서 비과세소득에 해당하는 소득은 다음과 같습니다:\n'
 '\n'
 '1. 「공익신탁법」에 따른 공익신탁의 이익\n'
 '\n'
 '2. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 '
 '제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 '
 '이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다). 이 경우 주택 수의 계산 및 주택임대소득의 산정 등 필요한 사항은 대통령령으로 '
 '정한다.\n'
 '\n'
 '3. 근로소득과 퇴직소득 중 다음 각 목의 어느 하나에 해당하는 소득\n'
 '\n'
 '   - ① 대통령령으로 정하는 복무 중인 병(兵)이 받는 급여\n'
 '   - ② 법률에 따라 동원된 사람이 그 동원 직장에서 받는 급여\n'
 '   - ③ 「산업재해보상보험법」에 따라 수급권자가 받는 요양급여, 휴업급여, 장해급여, 간병급여, 유족급여, 유족특별급여, '
 '장해특별급여, 장의비 또는 근로의 제공으로 인한 부상ㆍ질병ㆍ사망과 관련하여 근로자나 그 유족이 받는 배상ㆍ보상 또는 위자(慰藉)의 성질이 '
 '있는 급여\n'
 '   - ④ 「근로기준법」 또는 「선원법」에 따라 근로자ㆍ선원 및 그 유족이 받는 요양보상금, 휴업보상금, '
 '상병보상금(傷病補償frauen)과 그 밖의 금품\n'
 '   - ⑤ 「국군포로의 송환 및 대우 등에 관한 법률」에 따라 통계청장이 고시하는 한국표준직업분류에 따른 종교관련종사자(이하 '
 '"종교관련종사자"라 한다)가 받는 대통령령으로 정하는 학자금\n'
 '   - ⑥ 「국군포로의 송환 및 대우 등에 관한 법률」에 따라 국군포로가 받는 연금\n'
 '   - ⑦ 「상훈법」에 따른 훈장과 관련하여 받는 부상(副賞)이나 그 밖에 대통령령으로 정하는 상금과 부상\n'
 '\n'
 '4. 기타소득 중 다음 각 

In [15]:

try:
    llm2 = ChatOllama(
        model=CHAT_MODEL_NAME,
        temperature=0.1,
        num_predict=800
    )
    print("LLM 설정 완료")
except Exception as e:
    print(f"LLM 설정 실패: {e}")
    exit(1)

response = llm2.invoke(f"소득세법에서 비과세소득에 해당하는 소득은 무엇인가요?")
print('context 적용하지 않은 결과')
pprint(response.content)


LLM 설정 완료
context 적용하지 않은 결과
('소득세법에서 비과세소득에 대한 정의는 다음과 같습니다:\n'
 '\n'
 '1. **비과세소득**: 이는 "소득이 50만원 이상, 2억원 미만"이라는 조건을 만족하는 소득입니다.\n'
 '\n'
 '2. **소득세법에서 비과세소득은**:\n'
 '   - 50만원 이상의 소득\n'
 '   - 1억원 이하의 소득\n'
 '\n'
 '3. **비과세소득은**:\n'
 '   - 50만원 이상, 2억원 미만의 소득\n'
 '   - 1억원 이상, 2억원 미만의 소득\n'
 '\n'
 '4. **소득세법에서 비과세소득은**:\n'
 '   - 50만원 이상, 2억원 미만의 소득\n'
 '   - 1억원 이상, 2억원 미만의 소득\n'
 '\n'
 '이러한 조건을 만족하는 소득은 소득세를 면제할 수 있습니다. 이는 소득세법에서 비과세소득에 대한 정의를 명확하게 하고, 이를 통해 '
 '소득세를 효율적으로 관리하고, 불법적인 소득세를 피하기 위한 조치로 활용됩니다.\n'
 '\n'
 '이러한 조건을 만족하는 소득은 일반적으로 개인이나 기업 등에서 발생할 수 있습니다.')


### 개선한 Source - version1
* Retriever 검색방법 개선
    * search_type="mmr",  # 최대 다양성 검색
    * search_kwargs={"k": 6, "fetch_k": 10}  

In [16]:

from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings

from pprint import pprint

# 1. 데이터 로드 (기존과 동일)
loader = TextLoader("../data/taxinfo.txt", encoding="utf-8")
documents = loader.load()

# 2. 텍스트 분할 개선
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # 크기 증가
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""],  # 자연스러운 분할을 위한 구분자
    length_function=len,
    is_separator_regex=False,
)
split_docs = splitter.split_documents(documents)

# 3. 인덱싱 (벡터 저장)
embeddings_model = OllamaEmbeddings(model=EMBEDDING_MODEL_NAME)

vectorstore = FAISS.from_documents(
    documents=split_docs, 
    embedding=embeddings_model
)
#vectorstore = FAISS.from_documents(split_docs, OpenAIEmbeddings())

vectorstore.save_local("faiss_index")

# 4. 검색 개선
"""
    최대 다양성 검색(Maximum Marginal Relevance, MMR)
    MMR은 유사도가 높은 문서를 찾는 것을 넘어, 유사도와 다양성이라는 두 가지 기준을 모두 고려함
    - search_type="mmr": 검색 방식을 MMR로 지정합니다.
    - fetch_k: 일차적으로 질의와 유사한 문서 10개를 벡터 저장소에서 가져옵니다.
    - k: fetch_k로 가져온 10개의 문서 중에서 최종적으로 6개를 선택합니다. 6개를 선택할 때, MMR 알고리즘은 다음 두 가지를 고려함
        : 질의와의 유사도가 높고, 이미 선택된 다른 문서들과의 유사도가 낮은 (즉, 내용이 다양한) 문서
    * MMR의 작동 원리:
    - 질의와 가장 유사한 fetch_k개(10개)의 문서를 예비 후보군으로 가져옵니다.
    - 이 후보군 중에서 질의와 가장 유사한 문서 하나를 첫 번째 결과로 선택합니다.
    - 남은 후보군 중에서 질의와의 유사도는 높으면서 (관련성), 
      이미 선택된 문서들과의 유사도는 낮은 (다양성) 문서를 찾아 다음 결과로 추가함
    - 이 과정을 k개(6개)의 문서가 모두 선택될 때까지 반복합니다.        
"""
retriever = vectorstore.as_retriever(
    search_type="mmr",  # 최대 다양성 검색
    search_kwargs={"k": 6, "fetch_k": 10}  # 더 많은 결과 검색
)

# 5. 프롬프트 엔지니어링
def generate_prompt(query, context):
    return f"""다음은 소득세법 비과세소득 관련 조항입니다. 문맥을 고려하여 질문에 답변하세요.

[관련 조항]
{context}

[질문]
{query}

[답변 요구사항]
- 비과세소득 유형을 계층적으로 구분하여 설명
- 각 항목별 구체적인 조건 명시
- 법조문의 항, 호, 목 번호를 포함
- 500자 이내로 간결하게 요약"""

# 검색 및 응답 생성
query = "소득세법에서 비과세소득에 해당하는 소득은 무엇인가요?"
retrieved_docs = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

try:
    llm = ChatOllama(
        model=CHAT_MODEL_NAME,
        temperature=0.1,
        num_predict=800
    )
    print("LLM 설정 완료")
except Exception as e:
    print(f"LLM 설정 실패: {e}")
    exit(1)
    
response = llm.invoke(generate_prompt(query, context))

print('개선된 결과:')
pprint(response.content)

LLM 설정 완료
개선된 결과:
('소득세법에서 비과세소득에 해당하는 소득은 다음과 같습니다:\n'
 '\n'
 '1. **공익신탁**:\n'
 '   - 「공익신타피법」에 따른 공익신탰피의 이익: \n'
 '     - 논밭을 작물 생산에 이용하게 함으로써 발생하는 소득\n'
 '     - 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 '
 '임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 '
 '31일 이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다).\n'
 '\n'
 '2. **사업소득**:\n'
 '   - 「사업소득」: \n'
 '     - 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 '
 '임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 '
 '31일 이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다).\n'
 '\n'
 '3. **근로소득**:\n'
 '   - 「근로소득」: \n'
 '     - 「국가유공자 등 예우 및 지원에 관한 법률」 또는 「보훈보상대상자 지원에 관한 법률」에 따라 받는 보훈급여금ㆍ학습보조비, '
 '「북한이탈주민의 보호 및 정착지원에 관한 법률」에 따라 받는 정착금ㆍ보로금, 「전직대통령 예우에 관한 법률」에 따라 받는 연금.\n'
 '\n'
 '4. **교육기본법**:\n'
 '   - 「교육기본법」 제28조제1항에 따라 받는 장학금 중 대학생이 근로를 대가로 지급받는 장학금(「고등교육법」 제2조제1호부터 '
 '제4호까지의 규정에 따른 대학에 재학하는 대학생에 한정한다).\n'
 '\n'
 '5. **발명진흥법**:\n'
 '   - 「발명진흥법」 제2조제2호에 따른 종업원등(이하 이 조